# 07.4 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/04-arima-sarima/exercises.ipynb)

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)   # optimiser notices on short series; we check results ourselves
warnings.filterwarnings("ignore", message="Non-")      # "Non-stationary/Non-invertible starting parameters"
warnings.filterwarnings("ignore", category=FutureWarning, message="adfuller")   # upcoming API change notice

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

def sarima(train, h):
    """SARIMA(1,0,1)(0,1,0,52) with a constant: last year's week, corrected by recent errors."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # statsmodels start-value notices, not errors
        model = SARIMAX(train, order=(1, 0, 1), seasonal_order=(0, 1, 0, 52), trend="c").fit(disp=False)
    return model.forecast(h).values

## Exercise 1: by hand

(a) Difference `[200, 210, 205, 220, 230]` once. (b) Run AR(1) with c = 30, φ = 0.7 for three steps starting from y = 60. What is the long-run mean?

In [ ]:
# your code here

## Exercise 2: other orders

Backtest SARIMA with orders (1,0,0), (0,0,1) and (0,0,0) (all with seasonal order (0,1,0,52) and a constant). Which is best? What is (0,0,0)(0,1,0,52) with a constant, in plain words?

In [ ]:
# your code here

## Exercise 3: how good are the intervals?

For every backtest origin, fit `SARIMAX(1,0,1)(0,1,0,52)` and record whether each of the 5 actual weeks fell inside the 95% interval. What fraction did? Is the interval too wide, too narrow or about right?

In [ ]:
# your code here

## Exercise 4: monthly SARIMA

Fit `SARIMAX(monthly up to Oct 2011, order=(0,0,0), seasonal_order=(0,1,0,12), trend="c")` and forecast November 2011. Compare with the actual £1.16m and with the monthly Holt-Winters from 07.3 (+5.6%).

In [ ]:
# your code here